# B3: The event embedding space

Notebook A2 asked what a *language* model's embeddings know. This one asks the same question of
the encoder from notebook b2, the one trained on synthetic health records.

One thing to keep in mind: **the model was never shown an ontology.**
It has no drug classification, no disease hierarchy, no idea that a statin is a statin. It saw
integer sequences of events in patient timelines and was asked to fill in blanks. Anything
structured we find in this space, it learned from the sequences.

By the end you should be able to:

1. Query nearest neighbours and recognise clusters nobody told the model about.
2. **Measure** whether an embedding space has structure, rather than trusting a picture of it.
3. Compare projections and say what going down to two dimensions actually costs.

## Setup

In [ ]:
# --- Setup: runs locally and on Colab --------------------------------
# On Colab this installs what is missing and pulls the model and the data from
# the Hugging Face Hub. In a local checkout it finds both in the repository and
# installs nothing. torch and numpy are left alone: Colab's builds are
# CUDA-matched, and replacing them costs minutes and forces a runtime restart.
import pathlib
import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers==5.7.0",
            "pacmap>=0.9.1",
            "umap-learn>=0.5.12",
            "plotly>=6.9.0",
        ],
        check=True,
    )
    print("Colab: dependencies installed.")

import numpy as np
import pandas as pd
import torch

DATA_REPO = "carlomarxx/synthea-workshop-data"
MODEL_REPO = "carlomarxx/synthea-bert"


def resolve(local_path, repo_id, repo_type="dataset"):
    """Prefer the copy in this repository; fall back to the Hub if it is not here."""
    if pathlib.Path(local_path).exists():
        return str(local_path), "already in this repository"
    from huggingface_hub import snapshot_download

    return snapshot_download(repo_id, repo_type=repo_type), f"downloaded from {repo_id}"


DATA, data_source = resolve("../data/derived/workshop", DATA_REPO)
MODEL, model_source = resolve("../models/synthea-bert", MODEL_REPO, "model")

# event_bert.py sits beside the weights on the Hub, and in scripts/ locally.
sys.path[:0] = ["../scripts", MODEL]

print(f"data:  {DATA}\n       ({data_source})")
print(f"model: {MODEL}\n       ({model_source})")

from event_bert import EventBertForMaskedLM

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 62)

vocabulary = pd.read_csv(f"{DATA}/vocabulary.csv")
sequences = pd.read_parquet(f"{DATA}/sequences.parquet")
cohort = pd.read_parquet(f"{DATA}/cohort.parquet")

model = EventBertForMaskedLM.from_pretrained(MODEL, expected_vocab_size=len(vocabulary))
name_of = dict(zip(vocabulary.token_id, vocabulary.token))
id_of = dict(zip(vocabulary.token, vocabulary.token_id))

print(
    f"{len(vocabulary)} tokens, {sum(p.numel() for p in model.parameters()):,} parameters"
)


## 1. The embedding table

`get_input_embeddings()` returns the same kind of lookup table as notebook A2, just far
smaller: **1,036 x 128** instead of ModernBERT's 50,368 x 768. One row per token: every
event type, every background attribute, and the five special tokens.

The vocabulary file tells us what each row *is*, which is what lets us check the geometry
against something. Note that `source` (which table the code came from) is metadata we kept
for evaluation: **the model never saw it.**

In [ ]:
W = model.bert.get_input_embeddings().weight.detach().numpy().astype(np.float64)
print("embedding table:", W.shape)

events = vocabulary[vocabulary.kind == "event"].reset_index(drop=True)
event_ids = events.token_id.values
event_source = events.source.values

print(vocabulary.kind.value_counts().to_string())
print()
print(events.source.value_counts().to_string())

## 2. Look at it

Before any metric, the picture. **PaCMAP** squeezes the 985 event vectors from 128 dimensions
down to two, trying to keep near neighbours near. Colour is the source table each code came
from: `conditions`, `medications`, `procedures`, `immunizations`.

The model never saw that column. If the colours separate at all, the geometry worked it out
from co-occurrence in patient timelines and nothing else.

In [ ]:
import pacmap
import plotly.express as px


def unit(M):
    """Rows scaled to unit length, so a dot product is a cosine."""
    return M / np.linalg.norm(M, axis=1, keepdims=True)


# Section 6 measures what this projection costs and compares it against PCA and UMAP.
# Here it is just a first look, so we fit the one method and plot it.
XY = pacmap.PaCMAP(
    n_components=2, n_neighbors=10, distance="angular", random_state=7
).fit_transform(unit(W)[event_ids])

first_look = events.assign(x=XY[:, 0], y=XY[:, 1])

figure = px.scatter(
    first_look,
    x="x",
    y="y",
    color="source",
    hover_data={"token": True, "frequency": True, "x": False, "y": False},
    width=900,
    height=650,
    title="985 event types (PaCMAP). Colour = source table. Hover to read the codes.",
)
figure.update_traces(marker=dict(size=6, opacity=0.75))
figure.update_layout(xaxis_title=None, yaxis_title=None)
figure.show()

Zoom in and hover. Vaccines form their own island, the dental codes gather in a corner, and
the drugs are largely separate from the diagnoses.

Now the hard part. *"The clusters look sensible"* is the easiest claim in representation
learning to make and the hardest to defend, and this plot is two dimensions standing in for
128. Section 4 turns the picture into a number. Section 6 measures what the squeeze cost.

## 3. Nearest neighbours

The picture makes sense globally. Do the individual neighbourhoods?

Use `vocabulary.csv` to look up the names of the tokens.

In [ ]:
def nearest(token, k=8, space=None, kinds=("event",)):
    """The k most similar tokens to `token`, restricted to the given kinds."""
    space = W if space is None else space
    U = unit(space)
    scores = U @ U[id_of[token]]
    allowed = set(vocabulary[vocabulary.kind.isin(kinds)].token_id)
    order = [i for i in np.argsort(-scores) if i in allowed and name_of[i] != token][:k]
    return pd.DataFrame(
        {
            "similarity": [round(float(scores[i]), 3) for i in order],
            "token": [name_of[i] for i in order],
        }
    )


nearest("Diabetes mellitus type 2 (disorder)")

Diabetic neuropathy, ischemic heart disease, hyperglycemia, hypertriglyceridemia, diabetic
kidney disease, prediabetes, hyperlipidemia. That is a metabolic-syndrome cluster, and a
clinician would draw roughly the same picture.

The model does not know any of those words. `Diabetes mellitus type 2 (disorder)` is token
id 412 to it, and the only reason it sits near the others is that the same patients keep
having them.

In [ ]:
nearest("Stress (finding)")

Limited social contact, violence in the environment, victim of intimate partner abuse, social
isolation, drug misuse, severe anxiety, unhealthy alcohol use.

These are the **social determinants of health** codes, and they have found each other in a
space where nothing distinguished them from a knee replacement.

### ✏️ Exercise 1: play around with the neighbourhoods
1. Play around with the background tokens (see if the neighbourhood makes sense).


---
### (Optional Section) The clusters nobody labelled

Individual queries are suggestive. Clustering the whole space is the proof.

Below: k-means over the event vectors, ranked by how tight each cluster is, showing the
members closest to each centroid. Read the results before reading the next markdown cell.

In [ ]:
from sklearn.cluster import KMeans

N_CLUSTERS = 18
labels = KMeans(n_clusters=N_CLUSTERS, random_state=0, n_init=10).fit_predict(
    unit(W)[event_ids]
)
events["cluster"] = labels

U = unit(W)
similarity = U @ U.T
np.fill_diagonal(similarity, -9)


def coherence(members):
    """Mean pairwise cosine inside a group -- how tight the cluster is."""
    block = similarity[np.ix_(members, members)]
    return block[np.triu_indices(len(members), 1)].mean()


ranked = sorted(
    (
        (coherence(event_ids[labels == c]), c)
        for c in range(N_CLUSTERS)
        if (labels == c).sum() >= 6
    ),
    reverse=True,
)

for score, c in ranked[:4]:
    members = event_ids[labels == c]
    centroid = U[members].mean(0)
    ordered = members[np.argsort(-(U[members] @ centroid))]
    print(f"\ncluster {c}   n={len(members)}   coherence {score:.2f}")
    for t in ordered[:6]:
        print(f"    {name_of[int(t)][:76]}")

### Read what those are

Without being told, and with no drug ontology anywhere in the pipeline, the model has
separated:

- **statins and antihypertensives**: simvastatin, rosuvastatin, pravastatin next to ramipril,
  quinapril, irbesartan. The cardiovascular drawer.
- **oral contraceptives**: norethindrone and ethinyl estradiol packs, including brand-name
  variants that share no words with each other.
- **antiretrovirals**: lamivudine/zidovudine, efavirenz, tenofovir, lopinavir/ritonavir.
- **inhaled corticosteroids**: budesonide, fluticasone, mometasone, beclomethasone.

`{28 (norethindrone 0.35 MG Oral Tablet) } Pack [Camila 28 Day]` and
`{28 (norethindrone 0.35 MG Oral Tablet) } Pack [Errin 28 Day]` are, to the model, two
unrelated integers. It placed them together because the same kind of patient takes them.

This is the clearest demonstration in the workshop of what masked modelling buys you on
event data.

**(End of Optional Section)**

---

### ✏️ Exercise 2: find something the model gets wrong

`nearest()` takes any token in the vocabulary. Some queries come back clean, some come back
as noise.

Try a few of your own and find one where the neighbours are *not* sensible. Then look up its
frequency in `vocabulary`. Is it a rare code? Is it a code that means something vague, like
`Certification procedure (procedure)`?

A suggestion to start with, because it half-works: `Sepsis (disorder)` returns `Septic shock`
first, which is right, and then wanders off into fractures and gout.

In [ ]:
MY_TOKEN = "Sepsis (disorder)"  # <- change me

display(nearest(MY_TOKEN))
print()
print(
    vocabulary[vocabulary.token == MY_TOKEN][
        ["token_id", "source", "frequency"]
    ].to_string(index=False)
)

# TODO: pick two or three more tokens. Is neighbour quality mostly about frequency,
#       or about whether the code means something specific?

## 4. Is the space meaningful?

"These neighbours look sensible" is an anecdote. Here is a number.

For every event token, take its 10 nearest neighbours and ask what fraction come from the
**same source table**: conditions, medications, procedures, immunizations. The model never
saw that column, so if the geometry recovers it, the geometry is carrying real information.

Against what? The share you would get by picking neighbours at random, which is just how
common each source is. You implement it.

This is a deliberately *weak* test: it only asks whether medications sit near medications, not
whether statins sit near statins. Passing it is necessary, not sufficient, **but failing it
would mean the space is noise.**

In [ ]:
CHANCE = np.mean([events.source.value_counts(normalize=True)[s] for s in event_source])


def purity(space, k=10, index=None):
    """Fraction of each token's k nearest neighbours that share its source table."""
    index = event_ids if index is None else index
    U = unit(space)[index] if space.shape[1] > 2 else space[index]
    if space.shape[1] > 2:
        S = U @ U.T
    else:  # 2-D projections: use euclidean distance
        S = -((U[:, None, :] - U[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(S, -9e9)
    neighbours = np.argsort(-S, axis=1)[:, :k]
    return np.mean(
        [
            (event_source[row] == event_source[i]).mean()
            for i, row in enumerate(neighbours)
        ]
    )


print(f"chance                {CHANCE:.1%}")
print(f"raw 128-D embedding   {purity(W):.1%}")

**Question:** Is our space better than chance? 

<details>
<summary>
<b>Answer</b>
</summary>

More than twice chance. The space is not noise.

Is that just the common tokens carrying it, with the rare ones floating in the dark? Split by
frequency and see.

</details>


Not all neighbourhoods make sense. One explanation: rare tokens were seen less often during
training, so they are represented worse.


**Question**: Do rare tokens have "bad" representation, aka their neighbourhoods do not make any sense? Implement a test that would take rare tokens and estimate the purity of their neigborhoods. If our hypothesis is true, the purity will rise with the frequency of the token.

<details>
<summary><b>Answer</b></summary>

Barely a gradient: the rarest fifth of the vocabulary is only a few points behind the
commonest. Whatever is wrong with individual queries like `Sepsis`, it is not that rare tokens
were left untrained.

```
quintile = pd.qcut(
    events.frequency.rank(method="first"),
    5,
    labels=["rarest", "q2", "q3", "q4", "commonest"],
)

U = unit(W)[event_ids]
S = U @ U.T
np.fill_diagonal(S, -9)
per_token = np.array(
    [
        (event_source[row] == event_source[i]).mean()
        for i, row in enumerate(np.argsort(-S, axis=1)[:, :10])
    ]
)

pd.DataFrame(
    {"purity": per_token, "frequency": events.frequency, "quintile": quintile}
).groupby("quintile", observed=True).agg(
    n=("purity", "size"),
    median_frequency=("frequency", "median"),
    purity=("purity", "mean"),
).round(3)
```


</details>


In [ ]:
### implement the test
quintile = pd.qcut(
    events.frequency.rank(method="first"),
    5,
    labels=["rarest", "q2", "q3", "q4", "commonest"],
)

U = unit(W)[event_ids]
S = U @ U.T
np.fill_diagonal(S, -9)
per_token = np.array(
    [
        (event_source[row] == event_source[i]).mean()
        for i, row in enumerate(np.argsort(-S, axis=1)[:, :10])
    ]
)

pd.DataFrame(
    {"purity": per_token, "frequency": events.frequency, "quintile": quintile}
).groupby("quintile", observed=True).agg(
    n=("purity", "size"),
    median_frequency=("frequency", "median"),
    purity=("purity", "mean"),
).round(3)

## 5. A ruler out of anchor events

Nearest neighbours compare two tokens at a time. **Semantic projection** does something more
useful for social science: define a *direction* from two sets of anchor tokens, then score a
whole vocabulary along it.

Average a set of clearly social events, average a set of clearly clinical ones, subtract. The
result is a ruler. Project any token onto it and you get a number.

This is notebook A2's method on a different vocabulary. It comes from
[Grand et al. (2022)](https://www.nature.com/articles/s41562-022-01316-8) and, in its
best-known social-science application,
[Garg et al. (2018)](https://www.pnas.org/doi/10.1073/pnas.1720347115), who tracked a century
of gender and ethnic stereotypes by projecting occupation words in embeddings trained on
successive decades of text. Note what is being measured: **the corpus, not the world.** Here
the corpus is a simulator.

One thing is easier here than in A2. There are no word pieces: every event is exactly one
token, so `vec()` is a plain lookup and never averages fragments of a word.

In [ ]:
def vec(token):
    """The static row for one token. Every event is one token, so this is a plain lookup."""
    if token not in id_of:
        raise KeyError(f"{token!r} is not in the vocabulary")
    return W[id_of[token]]


def semantic_axis(pole_a, pole_b):
    """A direction in embedding space: mean of one pole minus mean of the other."""
    d = np.mean([vec(t) for t in pole_a], 0) - np.mean([vec(t) for t in pole_b], 0)
    return d / np.linalg.norm(d)


def project(tokens, axis):
    """Cosine of each token with the axis. Positive = pole A end, negative = pole B end."""
    scores = {t: float(vec(t) @ axis / np.linalg.norm(vec(t))) for t in tokens}
    return pd.Series(scores, name="projection").sort_values(ascending=False)

In [ ]:
# The two poles, and a list of tokens to score. Poles are the ruler; PROJECTED is what we
# measure with it, so the two must not overlap -- a token cannot help define the axis it is
# being scored on.
SOCIAL = [
    "Stress (finding)",
    "Social isolation (finding)",
    "Unemployed (finding)",
    "Housing unsatisfactory (finding)",
]
CLINICAL = [
    "Sepsis (disorder)",
    "Myocardial infarction (disorder)",
    "Pneumonia (disorder)",
    "Chronic kidney disease stage 1 (disorder)",
]
PROJECTED = [
    "Limited social contact (finding)",
    "Victim of intimate partner abuse (finding)",
    "Misuses drugs (finding)",
    "Reports of violence in the environment (finding)",
    "Body mass index 30+ - obesity (finding)",
    "Prediabetes (finding)",
    "Diabetes mellitus type 2 (disorder)",
    "Osteoarthritis of knee (disorder)",
    "Anemia (disorder)",
    "Chronic congestive heart failure (disorder)",
    "Chronic sinusitis (disorder)",
    "Acute bronchitis (disorder)",
]
assert not set(SOCIAL + CLINICAL) & set(PROJECTED), "an anchor cannot also be scored"

social_axis = semantic_axis(SOCIAL, CLINICAL)
social_scores = project(PROJECTED, social_axis)
print(social_scores.round(3).to_string())

The ordering is readable. Limited social contact, violence in the environment, intimate
partner abuse and drug misuse at the social end; congestive heart failure, osteoarthritis and
type 2 diabetes at the clinical end. Obesity and prediabetes sit in between, which is where a
social scientist would put them.

Before believing any of it, ask what a direction with **no** meaning would produce.

### The control that decides it

Projecting onto **random directions in 128-space** is not good enough. Embedding vectors do
not fill their space, they occupy a narrow cone
([Ethayarajh, 2019](https://arxiv.org/abs/1909.00512)). A direction built by subtracting two
real token vectors lies inside that cone and will spread projections further than an isotropic
random direction, *whatever* tokens you build it from. Comparing against isotropic noise
measures the cone, not the meaning.

The **matched** control builds its axis by the same recipe (average four tokens, subtract the
average of four others) but draws all eight at random. The cone effect is then present in both
arms and only the semantics differ.

Two pools, because the pool matters. Our anchors are all `conditions`, and `conditions` are
277 of the 985 event types. Random poles drawn from the whole vocabulary are mostly drugs and
procedures, which makes for an easier comparison than the real anchors faced. So we run the
control both ways, and the conditions-only pool is the one that has to hold.

In [ ]:
EVENT_POOL = events.token.tolist()
CONDITION_POOL = events[events.source == "conditions"].token.tolist()


def matched_null(tokens, pool, n=200, seed=0):
    """The same measurement, for `n` axes built from RANDOM anchor tokens.

    Each null axis uses the exact recipe of a real one -- average 4 tokens, subtract the
    average of 4 others -- except that all 8 are drawn at random from `pool` rather than
    chosen to mean something. A null axis therefore sits in the same narrow cone as
    `social_axis`, and the only thing it is missing is the semantics.
    """
    rng = np.random.default_rng(seed)
    return np.array(
        [
            project(tokens, semantic_axis(list(p[:4]), list(p[4:]))).std()
            for p in (rng.choice(pool, 8, replace=False) for _ in range(n))
        ]
    )


def report(name, scores, tokens):
    """Effect size of an axis against the matched nulls, in standard deviations."""
    spread = scores.std()
    print(f"{name}: spread of projections = {spread:.3f}")
    for label, pool in [
        ("all events", EVENT_POOL),
        ("conditions only", CONDITION_POOL),
    ]:
        null = matched_null(tokens, pool)
        z = (spread - null.mean()) / null.std()
        print(
            f"   vs random poles from {label:16s}"
            f" {null.mean():.3f} +/- {null.std():.3f}   z = {z:>5.1f}"
        )


report("social vs clinical", social_scores, PROJECTED)

### Example of the direction that does not work

The axis above separates these twelve tokens far better than a meaningless direction would,
under both pools. So the method works on this space.

Which makes the next one the interesting case. `Full-time employment` and `Part-time
employment` against `Unemployed` and `Housing unsatisfactory` is a **socioeconomic** ruler
built entirely out of event tokens, by the same recipe, scored on the same twelve tokens. It
is exactly the axis a social scientist would reach for.

In [ ]:
SECURE = ["Full-time employment (finding)", "Part-time employment (finding)"]
PRECARIOUS = ["Unemployed (finding)", "Housing unsatisfactory (finding)"]
assert not set(SECURE + PRECARIOUS) & set(PROJECTED), "an anchor cannot also be scored"

ses_axis = semantic_axis(SECURE, PRECARIOUS)
ses_scores = project(PROJECTED, ses_axis)
print(ses_scores.round(3).to_string())
print()
report("secure vs precarious", ses_scores, PROJECTED)

**Question**: the ordering above is not obviously nonsense. So why should we not report it?

<details>
<summary><b>Answer</b></summary>

Because the control says it is noise. The spread is 0.106 against a null of 0.105, so
**z = 0.0** on the all-events pool and **−0.6** on conditions only. A direction drawn at
random separates these twelve tokens exactly as well.

Read the ordering again knowing that. Violence in the environment, drug misuse and obesity
score on the *secure* side; congestive heart failure sits furthest into *precarious*. That is
not a weak version of the expected gradient, it is the wrong shape entirely, and it is what a
meaningless direction looks like when you insist on interpreting one.

Nothing in the numbers themselves warns you. The scores have the same range as the social
axis, they sort into a plausible-looking list, and the poles are four real, frequent,
well-trained tokens. Without the null you would have written a paragraph about it.

Same result for income, if you prefer background tokens as poles: `INCOME_B22`–`B25` against
`B01`–`B04` scores z = 0.7 against a null built from random *background* poles. Section 6
comes back to why.

> An axis is only as good as the control you ran against it. Build the null with the same
> recipe as the axis, or you are measuring the geometry rather than the meaning.

</details>

### ✏️ Exercise 3: build a ruler of your own

The cell below is prefilled with an acute-versus-chronic axis so it runs as-is. **Replace the
three lists** and see whether your axis survives the matched control.

Two rules, both of which the cell asserts for you:

1. No token may be both an anchor and a projected token.
2. Use tokens that are actually in the vocabulary. `nearest()` and `vocabulary.token` are
   there to browse.

If your axis comes back at z near zero, that is a result, not a failure. Say what you expected
it to measure and what the null says instead.

In [ ]:
# ---- replace these three lists -------------------------------------------------
MY_POLE_A = [
    "Sprain of ankle (disorder)",
    "Laceration of foot (disorder)",
    "Acute bronchitis (disorder)",
    "Concussion injury of brain (disorder)",
]
MY_POLE_B = [
    "Chronic kidney disease stage 3 (disorder)",
    "Chronic congestive heart failure (disorder)",
    "Chronic low back pain (finding)",
    "Chronic sinusitis (disorder)",
]
MY_TOKENS = [
    "Fracture of bone (disorder)",
    "Whiplash injury to neck (disorder)",
    "Otitis media (disorder)",
    "Streptococcal sore throat (disorder)",
    "Anemia (disorder)",
    "Prediabetes (finding)",
    "Diabetes mellitus type 2 (disorder)",
    "Osteoarthritis of knee (disorder)",
    "Chronic pain (finding)",
    "Body mass index 30+ - obesity (finding)",
]
# --------------------------------------------------------------------------------

assert not set(MY_POLE_A + MY_POLE_B) & set(MY_TOKENS), (
    "an anchor cannot also be scored"
)

my_scores = project(MY_TOKENS, semantic_axis(MY_POLE_A, MY_POLE_B))
print(my_scores.round(3).to_string())
print()
report("my axis", my_scores, MY_TOKENS)

## 6. (Optional) Choice of projection method

Section 2 opened with a PaCMAP plot and asked you to trust it. This is where we check.

Every projection throws something away. The honest way to pick one is to run the purity
metric from section 4 on its output: if the 2-D version scores far below the 128-D original,
the plot is hiding structure that is really there.

Three candidates:

- **PCA**: linear, deterministic, instant. Keeps the directions of largest variance.
- **PaCMAP**: preserves local *and* mid-range structure, and is fast.
- **UMAP**: the familiar one, and the slowest here.

In [ ]:
import time

from sklearn.decomposition import PCA
import pacmap
from umap import UMAP

X = unit(W)[event_ids]
projections = {}

for label, fit in [
    ("PCA", lambda: PCA(n_components=2, random_state=0).fit_transform(X)),
    (
        "PaCMAP",
        lambda: pacmap.PaCMAP(
            n_components=2, n_neighbors=10, distance="angular", random_state=7
        ).fit_transform(X),
    ),
    (
        "UMAP",
        lambda: UMAP(n_components=2, metric="cosine", random_state=7).fit_transform(X),
    ),
]:
    t0 = time.perf_counter()
    projections[label] = fit()
    elapsed = time.perf_counter() - t0
    # purity() indexes into the full vocabulary, so pad the 2-D coords back out
    padded = np.zeros((len(vocabulary), 2))
    padded[event_ids] = projections[label]
    print(f"{label:<8} purity {purity(padded):.1%}   [{elapsed:.1f}s]")

print(f"{'128-D':<8} purity {purity(W):.1%}   (the target)")
print(f"{'chance':<8}        {CHANCE:.1%}")

### PCA throws away a third of the structure

PCA drops to the mid-forties, barely above halfway between chance and the real thing. The
directions of largest variance in an embedding table are not the directions that separate
concepts, so a PCA scatter of this space is close to meaningless.

PaCMAP and UMAP both land near the 128-D number, so they are showing you something real.
PaCMAP is an order of magnitude faster and tends to keep mid-range structure that UMAP
distorts, so we use it below.

**A warning that applies to all three.** The coordinates are not reproducible across machines
or library versions. Read the *neighbourhoods*, never the positions, and never quote a
coordinate.

### ✏️ Explore the embedding space

In [ ]:
import plotly.express as px

frame = events.assign(
    x=projections["PaCMAP"][:, 0],
    y=projections["PaCMAP"][:, 1],
    cluster=labels.astype(str),
)

# Default to the labelled clusters: 985 undifferentiated dots read as confetti on a
# projector, which is exactly the failure mode section 6 is about.
TOP = [str(c) for _, c in ranked[:6]]
view = frame[frame.cluster.isin(TOP)]

figure = px.scatter(
    view,
    x="x",
    y="y",
    color="cluster",
    hover_data={
        "token": True,
        "source": True,
        "frequency": True,
        "x": False,
        "y": False,
    },
    width=900,
    height=650,
    title="The six tightest clusters (PaCMAP). Hover to read the codes.",
)
figure.update_traces(marker=dict(size=7))
figure.update_layout(xaxis_title=None, yaxis_title=None)
figure.show()

# To see everything instead, swap `view` for `frame` above and colour by "source".

## What to take away

1. **The model was never given an ontology.** Statins found statins, antiretrovirals found
   antiretrovirals, and contraceptive brands that share no words found each other, from
   co-occurrence in patient timelines and nothing else.
2. **Measure a space, do not admire it.** Neighbour purity of 70% against a 32% floor is a
   claim; "the clusters look sensible" is not.
3. **Two dimensions cost something, and you can find out how much.** PCA lost a third of the
   structure here. PaCMAP and UMAP did not. Run the metric before you trust the plot.
4. **An axis is only as good as its control.** A social-versus-clinical ruler separates
   twelve tokens at z = 4.8; a socioeconomic ruler built the same way, from equally real
   tokens, scores z = 0.0 and still sorts into a list you could narrate. Build the null with
   the same recipe as the axis.
Point 4 is the one to carry into notebook B4, where the same discipline decides whether a
prediction result is real.

### Going further

- [Ethayarajh (2019), *How contextual are contextualized word representations?*](https://arxiv.org/abs/1909.00512)
- [Mu & Viswanath (2018), *All-but-the-top*](https://arxiv.org/abs/1702.01417)
- [Wang et al. (2021), *Understanding how dimension reduction tools work*](https://jmlr.org/papers/v22/20-1061.html) -> the PaCMAP paper
- The life2vec concept space (Savcisens et al., 2024, Fig. 2) -> the same exercise on Danish
  register data, where the clusters are labour-market as well as clinical.